In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [ ]:
train_data = [
    ("영화가 정말 재미있다", "긍정"),
    ("최악의 영화였다", "부정"),
    ("배우 연기가 훌륭했다", "긍정"),
    ("스토리가 너무 지루했다", "부정"),
    ("정말 감동적이었다", "긍정"),
    ("시간 낭비였다", "부정"),
]

test_reviews = [
    "이 영화는 정말 멋있었어!",
    "완전히 실망했다",
    "그냥 볼만했다"
]

In [ ]:
# zero-shot prompt
zero_shot_prompt = '''
다음 영화 리뷰의 감성을 분석하세요

리뷰:'이 영화는 정말 멋있었어!'
감성:
'''
one_shot_prompt = '''
다음 영화 리뷰의 감성을 분석하세요.

리뷰: "영화가 정말 재미있었어요!"
감성: 긍정

리뷰: "이 영화는 정말 멋있었어!"
감성:
'''
few_shot_prompt = '''
다음 영화 리뷰의 감성을 분석하세요.

리뷰: "영화가 정말 재미있었어요!"
감성: 긍정

리뷰: "완전히 실망했어요."
감성: 부정

리뷰: "그런대로 괜찮네요."
감성: 중립

리뷰: "이 영화는 정말 멋있었어!"
감성:
'''

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
# %pip install accelerate

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
print(model)

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def generate_sentiment(prompt, max_new_tokens=20):
    message = [
        {
            "role" : "system",
            'content' : '너는 영화 리뷰 감성을 긍정, 부정, 중립중 하나로만 분류하는 분류기입니다.'
        },
        {
            'role':'user',
            'content':prompt
        }
    ]
    text = tokenizer.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text,return_tensors='pt').to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0][ inputs['input_ids'].shape[-1] : ]
    return tokenizer.decode(generated_ids,skip_special_tokens=True).strip()

In [ ]:
results = {
    'Zero-shot' : generate_sentiment(zero_shot_prompt),
    'One-shot' : generate_sentiment(one_shot_prompt),
    'Few-shot' : generate_sentiment(few_shot_prompt),
}
results